# Integratore, media temporale, riduzione rumore alta frequenza, smoothing #

## Cosa fa un integratore ##

Abbiamo dimostrato che, per un periodo $T$ del segnale (periodico) in ingresso $V_{in}(t)$ sufficientemente minore del tempo caratteristico $\tau = RC$ del circuito, il segnale in uscita è 
$$V_{out}(t) \simeq \frac{1}{\tau}\int V_{in}(t^{\prime}) \rm{d}t^{\prime}\;.$$
Nonostante la denominazione "integratore", e tenendo conto che l'intervallo temporale di integrazione è dell'ordine di $\tau$, la presenza del termine $\tau$ a dividere rende l'operazione molto simile a una **media temporale** (più propriamente, la media aritmetica di una funzione).

Oltre ad avere tantissime applicazioni per il trattamento di segnali dipendenti dal tempo, questa operazione è rilevante anche nell'ambito della **misura** di segnali elettrici. Infatti mediando temporalmente:
- si riduce la sensibilità rispetto a variazioni rapide (rispetto a $\tau$);
- si esalta quella rispetto a variazioni lente (sempre rispetto a $\tau$).

Lo script qui di seguito considera un segnale periodico di periodo $T$ all'ingresso di un integratore con tempo caratteristico $\tau$: il rapporto $\tau/T$ può essere variato agendo su uno slider. Per simulare in qualche modo una situazione realistica, al segnale periodico è sovrapposto un "rumore" costituito da una perturbazione sinusoidale ad "alta frequenza" (di default 23.4 volte maggiore della frequenza del segnale - numero scelto a caso!) e da una fluttuazione stocastica di piccola ampiezza.

A differenza di quanto faremo/farete per l'esercizio/relazione (obbligatoria), il meccanismo di integrazione è qui ottenuto in maniera molto brutale (e **poco corretta**) calcolando la *media mobile* del segnale in ingresso, eseguita su una finestra temporale pari a $\tau$: potete facilmente vedere come è scritto l'algoritmo. 

Potete anche facilmente vedere quali sono gli effetti: per scelte opportune di $\tau$ il segnale viene *smussato*, con una apparente sopressione del rumore ad alta frequenza e delle fluttuazioni. Aumentando ulteriormente $\tau$ e ponendolo dell'ordine di una frazione di $T$, l'integratore comincia a modificare anche il segnale in maniera evidente, deformandolo e attenuaandolo come da aspettative.

In [5]:
from ipywidgets import interact
import ipywidgets as widgets
import numpy as np
import pylab as py

wavef = 1 # seleziona 1 per quadra, 2 per triangolare, 3 per sinusoidale
nofpoints = 5000 # numero di punti degli array
nofperiods = 5 # numero di periodi graficati del segnale
hfnoisefreq = 23.4 # frequenza rumore in unità della frequenza del segnale 
amplnoise = 0.2 # ampiezza rumore alta frequenza in unità dell'ampiezza del segnale
sfashf = 0.3 # sfasamento rumore alta frequenza
randomampl = 0.4 # ampiezza massima rumore random

period = nofpoints/nofperiods # punti in un singolo periodo
omega = 2*np.pi # frequenza segnale
omegahf = omega*hfnoisefreq # frequenza rumore alta frequenza
tarray = np.linspace(0,nofperiods,nofpoints) # array dei tempi

def func(t): # definisce la funzione del segnale
    if (wavef==1):
        return np.sign(np.sin(omega*t))
    if (wavef==2):
        return (2/np.pi)*np.asin(np.sin(omega*t))
    if (wavef==3):
        return np.sin(omega*t)

def hfnoise(t): # definisce la funzione sinusoidale del rumore ad alta frequenza
    return amplnoise*np.sin(omegahf*t)

rng = np.random.default_rng(seed=42) # crea un po' di casino (rumore casuale)
spikelee = randomampl*(rng.random(size=nofpoints)-0.5)

inparray = func(tarray) + hfnoise(tarray) + spikelee # questo è il segnale in ingresso

def plotta(tausut):
    k = int(period*tausut) # stabilisce su quanti punti fare la somma mobile
    outarray = []
    for i in range (k,nofpoints): # crea il segnale di uscita mettendo in ogni punto la somma dei k punti precedenti (divisa per il "tempo di integrazione" tau)
        outarray.append(np.sum(inparray[i-k:i]/k))
    tarrayout = np.linspace(tausut,nofperiods,nofpoints-k) # crea l'array che fa da asse x per l'uscita
    py.figure(figsize=(14,4))
    py.plot(tarray,inparray)
    py.plot(tarrayout,outarray,color='red')
    py.xlabel('Time  [T]',fontsize='14')
    py.ylabel('Signal  [arb.un.]',fontsize='14')
    py.legend(['input','output'],fontsize='12')

# fa il grafico interattivo con lo slider
interact(
    plotta,
    tausut = widgets.FloatSlider(value=5e-3, min=2e-3, max=0.3, step=1e-3,description='$\\tau/T$',readout_format='.3f')
    )   

interactive(children=(FloatSlider(value=0.005, description='$\\tau/T$', max=0.3, min=0.002, readout_format='.3…

<function __main__.plotta(tausut)>

## Smoothing (Savitzky-Golay) ##

Tecnicamente, il calcolo della media mobile effettuato nello script precedente rappresenta un'operazione di **smoothing** del segnale. Questa è una operazione impiegata molto frequentemente allo scopo di sopprimere, o attenuare, le variazioni "rapide" del segnale, per esempio dovute a rumore ad alta frequenza o fluttuazioni rapide.

Esistono diversi algoritmi in grado di smussare un array in modo più raffinato rispetto alla (bruttissima!) procedura dello script precedente. Uno degli algoritmi più comuni prende il nome da **Savitzky-Golay**: potete googolare per vederne i dettagli.

Il pacchetto *signal* della libreria *scipy* di Python include una versione di questo algoritmo. Lo script che segue è identico al precedente, salvo per l'applicazione, molto compatta e sufficientemente elegante, dello smoothing alla Savitzky-Golay al *primo ordine* (potete provare a cambiare l'ordine, cioè il grado del polinomio interpolante). Il risultato è anche, ovviamente, molto simile a quanto trovato in precedenza.

**Warning and disclaimer**: l'applicazione dello smoothing rappresenta a tutti gli effetti una **manipolazione** dei dati. Pertanto essa può essere tollerata solo se fisicamente giustificata, per esempio per la (verificata) presenza di rumore ad alta frequenza e fluttuazioni, e deve essere **sempre chiaramente dichiarata**!

In [6]:
from ipywidgets import interact
import ipywidgets as widgets
import numpy as np
import pylab as py
from scipy.signal import savgol_filter # pacchetto che contiene il filtro

wavef = 3 # seleziona 1 per quadra, 2 per triangolare, 3 per sinusoidale
nofpoints = 5000 # numero di punti degli array
nofperiods = 5 # numero di periodi graficati del segnale
hfnoisefreq = 23.4 # frequenza rumore in unità della frequenza del segnale 
amplnoise = 0.2 # ampiezza rumore alta frequenza in unità dell'ampiezza del segnale
sfashf = 0.3 # sfasamento rumore alta frequenza
randomampl = 0.4 # ampiezza massima rumore random

period = nofpoints/nofperiods # punti in un singolo periodo
omega = 2*np.pi # frequenza segnale
omegahf = omega*hfnoisefreq # frequenza rumore alta frequenza
tarray = np.linspace(0,nofperiods,nofpoints) # array dei tempi

def func(t): # definisce la funzione del segnale
    if (wavef==1):
        return np.sign(np.sin(omega*t))
    if (wavef==2):
        return (2/np.pi)*np.asin(np.sin(omega*t))
    if (wavef==3):
        return np.sin(omega*t)

def hfnoise(t): # definisce la funzione sinusoidale del rumore ad alta frequenza
    return amplnoise*np.sin(omegahf*t)

rng = np.random.default_rng(seed=42) # crea un po' di casino (rumore casuale)
spikelee = randomampl*(rng.random(size=nofpoints)-0.5)

inparray = func(tarray) + hfnoise(tarray) + spikelee # questo è il segnale in ingresso

def plotta(tausut):
    k = int(period*tausut) # stabilisce su quanti punti fare la somma mobile
    outarray = savgol_filter(inparray, k, 1) # applica Savitzky-Golay lineare su finestra lunga k
    # qui sotto si tolgono (mettono pari a NaN) punti all'inizio e alla fine dell'array
    #che non sono correttamente definiti per la mancanza di adeguate condizioni al contorno
    outarray[:k] = np.nan
    outarray[nofpoints-k:] = np.nan
    py.figure(figsize=(14,4))
    py.plot(tarray,inparray)
    py.plot(tarray,outarray,color='red')
    py.xlabel('Time  [T]',fontsize='14')
    py.ylabel('Signal  [arb.un.]',fontsize='14')
    py.legend(['input','output'],fontsize='12')

# fa il grafico interattivo con lo slider
interact(
    plotta,
    tausut = widgets.FloatSlider(value=5e-3, min=2e-3, max=0.5, step=1e-3,description='$\\tau/T$',readout_format='.3f')
    )   

interactive(children=(FloatSlider(value=0.005, description='$\\tau/T$', max=0.5, min=0.002, readout_format='.3…

<function __main__.plotta(tausut)>

# Flicker noise e altri metodi #

Quanto abbiamo mostrato qui sopra indica che, a patto di trovarsi nelle condizioni opportune, un segnale con rumore ad alta frequenza e fluttuazioni rapide può essere ripulito attraverso integrazione temporale.

Integrare temporalmente significa far durare più a lungo la misura, cioè ridurre artificialmente la "prontezza" dello strumento. A livello discorsivo questo è in accordo con la constatazione (generalmente) valida che allungare il tempo di misura migliora il rapporto segnale/rumore (S/N).

A parte che, così facendo, si perde la possibilità di seguire ogni variazione rapida del segnale, aumentare il tempo di misura ha anche un altro effetto negativo. Nella classificazione convenzionale dei rumori che si fa in fisica, uno, detto **flicker noise** o **rumore rosa** o **rumore 1/f** (con f si intende la frequenza, cioè il reciproco del tempo di misura), diventa sempre più rilevante integrando per tempi lunghi. In genere esso è associato a variazioni (*drift*) su scale temporali lunghe, per esempio effetti legati a variazioni di temperatura, vibrazioni meccaniche, rumori acustici o sismici. 

Morale della favola è che, molto spesso, integrando troppo si finisce per esaltare una tipologia di rumore che è a bassa, o bassissima, frequenza. 

Altra morale della favola è che esistono metodi, detti generalmente di **detezione sincrona** (per esempio, il **lock-in**) che permettono di salvare capra e cavoli, cioè di attenuare il rumore ad alta frequenza senza dover necessariamente integrare per tempi (molto) lunghi. 

Magari ne parleremo in futuro!

# Forme imperfette #

Le forme d'onda prodotte dal generatore escono da uno strumento **reale**. L'aggettivo ha numerose implicazioni, per esempio che la resistenza di uscita è finita (50 ohm), oppure che la frequenza non può essere definita con precisione arbitrariamente alta come nel mondo fatato dei modelli matematici. C'è un'ulteriore possibile implicazione: la "forma" stessa può essere anche sensibilmente diversa da quella, banalissima, del modello matematico. 

Naturalmente, trattandosi di uno strumento per uso scientifico, tutte le deviazioni della realtà sono riportate nel manuale, dove in genere è espressa la **tolleranza** rispetto al comportamento "ideale".

Prendiamo il caso più semplice di una forma d'onda quadra che supponiamo di rendere *alternata*, cioè a media temporale nulla su un periodon (offset nullo e duty-cycle 50:50). Molti di voi si sono accorti che, se una tale forma d'onda viene inviata a un integratore, l'uscita acquisisce un offset non nullo.

Ci sono almeno due possibili motivi:
- un piccolo offset potrebbe già essere presente nella forma d'onda. Questo piccolo offset diventa visibile quando si aumenta la sensibilità della misura (un integratore che integra attenua la componente variabile nel tempo del segnale e quindi è normale che la sensibilità venga aumentata per visualizzare l'uscita);
- la forma d'onda non è *idealmente* simmetrica.

Analizziamo questo secondo punto. Stando al manuale, l'asimmetria tollerata è del $\pm$2%. Questo potrebbe significare, ad esempio, che il *duty-cycle* dell'onda quadra si discosta dal rapporto (ideale) 50:50. La forma d'onda non è più, quindi, alternata e il processo di integrazione porta a un offset diverso da zero.

Lo script che segue, che fa uso di alcune (potenzialmente interessanti) funzionalità del pacchetto *scipy*, dimostra quanto affermato. Nello script l'operazione di integrazione è effettuata tramite smoothing (Savitzky-Golay) su  una finestra temporale che di default è posta a 5 periodi, così da simulare il comportamento di un circuito che integra in modo molto "efficace". Se all'ingresso viene mandata una forma quadra non simmetrica (il duty-cycle si può variare con uno slider), all'uscita si ha un offset diverso da zero (osservate la diversa scala per i due assi verticali!).

In [7]:
from ipywidgets import interact
import ipywidgets as widgets
import numpy as np
import pylab as py
from scipy.signal import * # importa tutto il contenuto di scipy.signal
import scipy # e anche il pacchetto scipy

wavef = 1 # seleziona 1 per quadra, 2 per triangolare, 3 per sinusoidale
nofpoints = 5000 # numero di punti degli array
nofperiods = 20 # numero di periodi graficati del segnale
nofsmoothedperiods = 5 # intervallo di smoothing espresso in unità di periodo

period = nofpoints/nofperiods # punti in un singolo periodo
omega = 2*np.pi # frequenza segnale
tarray = np.linspace(0,nofperiods,nofpoints) # array dei tempi

def plotta(dutycycle):
    inparray = scipy.signal.square(omega*tarray,duty=dutycycle/100)
    k = int(period*nofsmoothedperiods) # stabilisce su quanti punti fare la somma mobile
    outarray = savgol_filter(inparray, k, 1) # applica Savitzky-Golay lineare su finestra lunga quanto l'intero array in ingresso
    # qui sotto si tolgono (mettono pari a NaN) punti all'inizio e alla fine dell'array
    #che non sono correttamente definiti per la mancanza di adeguate condizioni al contorno
    outarray[:k] = np.nan
    outarray[nofpoints-k:] = np.nan
    fig, ax1 = py.subplots(figsize=(14,4)) 
    ax1.plot(tarray,inparray)
    ax1.set_xlabel('Time  [T]',fontsize='14')
    ax1.set_ylabel('Input signal  [arb.un.]',fontsize='14',color='blue')
    ax2 = ax1.twinx() # serve per creare il doppio asse verticale
    ax2.plot(tarray,outarray,color='red')
    ax2.set_ylim(-.2,.2)
    ax2.set_ylabel('Output signal  [arb.un]',fontsize='14',color='red')

# fa il grafico interattivo con lo slider
interact(
    plotta,
    dutycycle = widgets.FloatSlider(value=48, min=42, max=58, step=2,description='Duty-cycle',readout_format='.0f')
    )   

interactive(children=(FloatSlider(value=48.0, description='Duty-cycle', max=58.0, min=42.0, readout_format='.0…

<function __main__.plotta(dutycycle)>